In [1]:
import pandas as pd
import numpy as np
import requests
import time

In [2]:
path_folder = "../data/processed/"
path_claim_clusters = "ds_claims_clustered.xlsx"

In [3]:
ds_claim_w_cluster = pd.read_excel(
    path_folder+path_claim_clusters,
    sheet_name=0
)

In [4]:
ds_claim_w_cluster = ds_claim_w_cluster[
    ["Claim Number", "Lat", "Lon", "Cluster","Centroid_Lat","Centroid_Lon"]
    ]

In [5]:
OSRM_BASE = "https://router.project-osrm.org"

def osrm_table_batch(origins_latlon, destinations_latlon, sleep_s=0.2, timeout=60, annotations="duration"):
    coords = []
    src_idx = []
    dst_idx = []

    for (lat, lon) in origins_latlon:
        src_idx.append(len(coords))
        coords.append((lon, lat))

    for (lat, lon) in destinations_latlon:
        dst_idx.append(len(coords))
        coords.append((lon, lat))

    coords_str = ";".join([f"{lon:.6f},{lat:.6f}" for lon, lat in coords])

    url = f"{OSRM_BASE}/table/v1/driving/{coords_str}"
    params = {
        "sources": ";".join(map(str, src_idx)),
        "destinations": ";".join(map(str, dst_idx)),
        "annotations": annotations,
    }

    r = requests.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    data = r.json()
    if data.get("code") != "Ok":
        raise RuntimeError(f"OSRM error: {data}")

    out = {}
    if "duration" in annotations:
        out["durations"] = np.array(
            [[np.nan if x is None else float(x) for x in row] for row in data["durations"]],
            dtype=float
        )
    if "distance" in annotations:
        out["distances"] = np.array(
            [[np.nan if x is None else float(x) for x in row] for row in data["distances"]],
            dtype=float
        )

    if sleep_s:
        time.sleep(sleep_s)

    return out

In [6]:
def _chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def build_claim_to_centroid_osrm(
    claims_w_centroid: pd.DataFrame,
    claim_id_col="Claim Number",
    claim_lat_col="Lat",
    claim_lon_col="Lon",
    cluster_col="Cluster",
    centroid_lat_col="Centroid_Lat",
    centroid_lon_col="Centroid_Lon",
    O_BATCH=50,          # origins batch size (claims)
    sleep_s=0.2,
    timeout=60,
    return_distance=True
):
    needed = [claim_id_col, claim_lat_col, claim_lon_col, cluster_col, centroid_lat_col, centroid_lon_col]
    df = claims_w_centroid[needed].dropna().copy()

    results = []

    annotations = "duration,distance" if return_distance else "duration"

    for cluster_id, g in df.groupby(cluster_col, sort=False):
        # One destination: this cluster's centroid
        cen_lat = float(g[centroid_lat_col].iloc[0])
        cen_lon = float(g[centroid_lon_col].iloc[0])
        dest = [(cen_lat, cen_lon)]

        g = g.reset_index(drop=True)

        for idx_batch in _chunks(list(range(len(g))), O_BATCH):
            gb = g.iloc[idx_batch]

            origins = list(zip(
                gb[claim_lat_col].to_numpy(dtype=float),
                gb[claim_lon_col].to_numpy(dtype=float),
            ))

            out = osrm_table_batch(
                origins_latlon=origins,
                destinations_latlon=dest,
                sleep_s=sleep_s,
                timeout=timeout,
                annotations=annotations
            )

            dur = out["durations"][:, 0]  # (O,1) -> (O,)
            if return_distance:
                dist_m = out["distances"][:, 0]
                dist_miles = dist_m / 1609.344

            for i, row in gb.iterrows():
                if return_distance:
                    results.append((row[claim_id_col], row[cluster_col], dur[list(gb.index).index(i)], dist_miles[list(gb.index).index(i)]))
                else:
                    results.append((row[claim_id_col], row[cluster_col], dur[list(gb.index).index(i)]))

    if return_distance:
        return pd.DataFrame(results, columns=[claim_id_col, cluster_col, "drive_seconds", "drive_miles"])
    return pd.DataFrame(results, columns=[claim_id_col, cluster_col, "drive_seconds"])

In [7]:
time_to_centroid = build_claim_to_centroid_osrm(
    claims_w_centroid=ds_claim_w_cluster,
    O_BATCH=50,
    sleep_s=0.2,
    return_distance=True
)

In [8]:
time_to_centroid["drive_seconds"] = time_to_centroid["drive_seconds"] * 2
time_to_centroid["drive_miles"] = time_to_centroid["drive_miles"] * 2

In [9]:
time_to_centroid.to_excel(
    "../data/processed/ds_timeMatrix.xlsx",
    index=False
)

In [10]:
cluster_max = (
    time_to_centroid
    .groupby("Cluster", as_index=False)
    .agg(
        Claims=("Claim Number", "count"),
        Max_Drive_Seconds=("drive_seconds", "max"),
        Max_Drive_Miles=("drive_miles", "max"),
    )
)

cluster_max["Max_Drive_Min"] = cluster_max["Max_Drive_Seconds"] / 60
cluster_max

,Cluster,Claims,Max_Drive_Seconds,Max_Drive_Miles,Max_Drive_Min
0,0.0,202,11813.2,168.658534,196.886667
1,1.0,72,13927.4,150.534255,232.123333
2,2.0,54,18381.8,221.054666,306.363333
3,3.0,44,13071.2,146.091451,217.853333
4,4.0,29,13030.6,147.459710,217.176667
5,5.0,52,13210.0,160.347073,220.166667
6,6.0,60,8918.0,113.330152,148.633333
7,7.0,87,13533.2,183.095970,225.553333
8,8.0,81,17133.8,241.187838,285.563333
9,9.0,43,14627.8,176.473892,243.796667


In [2]:
import pandas as pd
import numpy as np

# clustered file with mathematical estimates
clustered_claims = pd.read_excel("../data/processed/ds_claims_clustered.xlsx", sheet_name="Clustered Claims")

# osrm matrix
osrm_tm = pd.read_excel("../data/processed/ds_timeMatrix.xlsx")

# keep needed cols
math_tm = clustered_claims[[
    "Claim Number", "Cluster",
    "Straight_Line_Miles", "Est_Road_Miles", "Est_Drive_Time_Min"
]].copy()

cmp = math_tm.merge(
    osrm_tm[["Claim Number", "Cluster", "drive_seconds", "drive_miles"]],
    on=["Claim Number", "Cluster"],
    how="inner"
)

# OSRM one-way and round-trip
cmp["OSRM_Drive_Min_RoundTrip"] = cmp["drive_seconds"] / 60
cmp["OSRM_Drive_Miles_RoundTrip"] = cmp["drive_miles"]

# ratios: mathematical / OSRM roundtrip
cmp["Miles_Ratio_Math_to_OSRM"] = cmp["Est_Road_Miles"] / cmp["OSRM_Drive_Miles_RoundTrip"]
cmp["Time_Ratio_Math_to_OSRM"] = cmp["Est_Drive_Time_Min"] / cmp["OSRM_Drive_Min_RoundTrip"]

# errors
cmp["Miles_Diff"] = cmp["Est_Road_Miles"] - cmp["OSRM_Drive_Miles_RoundTrip"]
cmp["Time_Diff_Min"] = cmp["Est_Drive_Time_Min"] - cmp["OSRM_Drive_Min_RoundTrip"]

cmp["Est_Drive_Time_Min"] = cmp["Est_Drive_Time_Min"].round(2)
cmp["Est_Road_Miles"] = cmp["Est_Road_Miles"].round(2)

summary = pd.DataFrame({
    "metric": [
        "n_rows",
        "avg_math_road_miles",
        "avg_osrm_roundtrip_miles",
        "avg_math_drive_min",
        "avg_osrm_roundtrip_min",
        "median_miles_ratio_math_to_osrm",
        "median_time_ratio_math_to_osrm",
        "mean_miles_ratio_math_to_osrm",
        "mean_time_ratio_math_to_osrm"
    ],
    "value": [
        len(cmp),
        cmp["Est_Road_Miles"].mean(),
        cmp["OSRM_Drive_Miles_RoundTrip"].mean(),
        cmp["Est_Drive_Time_Min"].mean(),
        cmp["OSRM_Drive_Min_RoundTrip"].mean(),
        cmp["Miles_Ratio_Math_to_OSRM"].median(),
        cmp["Time_Ratio_Math_to_OSRM"].median(),
        cmp["Miles_Ratio_Math_to_OSRM"].mean(),
        cmp["Time_Ratio_Math_to_OSRM"].mean()
    ]
})

display(summary)

# by-cluster comparison
cluster_compare = (
    cmp.groupby("Cluster", as_index=False)
    .agg(
        Claims=("Claim Number", "count"),
        Math_Miles=("Est_Road_Miles", "mean"),
        OSRM_Miles_RT=("OSRM_Drive_Miles_RoundTrip", "mean"),
        Math_Min=("Est_Drive_Time_Min", "mean"),
        OSRM_Min_RT=("OSRM_Drive_Min_RoundTrip", "mean"),
        Median_Miles_Ratio=("Miles_Ratio_Math_to_OSRM", "median"),
        Median_Time_Ratio=("Time_Ratio_Math_to_OSRM", "median")
    )
)

display(cluster_compare)

# save if needed
cmp.to_excel("../data/processed/ds_time_compare_math_vs_osrm.xlsx", index=False)
cluster_compare.to_excel("../data/processed/ds_time_compare_by_cluster.xlsx", index=False)

,metric,value
0,n_rows,1062.000000
1,avg_math_road_miles,48.331883
2,avg_osrm_roundtrip_miles,48.837869
3,avg_math_drive_min,72.497646
4,avg_osrm_roundtrip_min,77.285766
5,median_miles_ratio_math_to_osrm,0.993896
6,median_time_ratio_math_to_osrm,0.869554
7,mean_miles_ratio_math_to_osrm,inf
8,mean_time_ratio_math_to_osrm,inf


,Cluster,Claims,Math_Miles,OSRM_Miles_RT,Math_Min,OSRM_Min_RT,Median_Miles_Ratio,Median_Time_Ratio
0,0,202,21.831832,21.890889,32.747723,36.291205,1.001243,0.842744
1,1,72,69.444167,66.265169,104.167222,109.805972,1.061707,0.908637
2,2,54,53.335926,55.055156,80.003889,81.106852,0.949771,0.882161
3,3,44,73.639773,70.962338,110.459091,98.330530,1.027826,1.101046
4,4,29,74.036207,80.860533,111.058966,143.075517,0.926239,0.757021
5,5,52,70.005192,63.788564,105.008846,89.786795,1.102988,1.152223
6,6,60,54.748500,53.393113,82.123000,78.529556,1.030832,1.033007
7,7,87,56.889885,59.168431,85.334598,88.827739,0.972094,0.937466
8,8,81,21.178025,23.566856,31.767531,35.857078,0.918653,0.832181
9,9,43,73.247209,74.027427,109.871860,100.653876,0.969890,1.021319
